# Build LineageOS 23.2 (x86 ISO) + GApps + KernelSU
Cara ini mengatasi masalah penyimpanan Colab yang kecil dengan mengunduh *source code* dan menyimpannya langsung ke Google Drive Anda.

**⚠️ PERINGATAN PENTING SEBELUM MEMULAI:**
1. **Butuh Storage Ekstra Besar:** Anda WAJIB memiliki kapasitas Google Drive kosong minimal **150 GB - 200 GB**. Akun Google gratis (yang hanya 15GB) pasti akan penuh dan error. Anda harus menggunakan akun Google One (berbayar).
2. **Proses Sangat Lambat:** Karena Google Drive bukan penyimpanan lokal fisik (melainkan *cloud sync*), kecepatan baca/tulis (*I/O*) jutaan file Android sangatlah lambat. Proses `repo sync` dan kompilasi yang biasanya selesai dalam beberapa jam, bisa memakan waktu **berhari-hari**.

In [ ]:
# 1. Hubungkan (Mount) Google Drive Anda ke Colab
# Akan muncul pop-up meminta izin akses akun Google Anda.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# 2. Instal Dependensi (Alat untuk kompilasi Android)
sudo apt update
sudo apt install -y bc bison build-essential ccache curl flex g++-multilib gcc-multilib git gnupg gperf imagemagick lib32ncurses5-dev lib32readline-dev lib32z1-dev liblz4-tool libncurses5 libncurses5-dev libsdl1.2-dev libssl-dev libxml2 libxml2-utils lzop pngcrush rsync schedtool squashfs-tools xsltproc zip zlib1g-dev

In [ ]:
%%bash
# 3. Pasang 'repo' dan konfigurasi Git
mkdir -p ~/bin
curl https://storage.googleapis.com/git-repo-downloads/repo > ~/bin/repo
chmod a+x ~/bin/repo
sudo ln -sf ~/bin/repo /usr/bin/repo

git config --global user.name "Colab BuildBot"
git config --global user.email "colab@example.com"

In [ ]:
%%bash
# 4. Unduh Source Code LineageOS & Google Apps & KernelSU
mkdir -p '/content/drive/MyDrive/lineageos_x86'
cd '/content/drive/MyDrive/lineageos_x86'

# Inisialisasi dan sinkronisasi OS
repo init -u https://github.com/LineageOS/android.git -b lineage-23.2 --depth=1
repo sync -c -j$(nproc --all) --force-sync --no-clone-bundle --no-tags

# Kloning repositori Google Services resmi LineageOS (MindTheGapps)
git clone https://gitlab.com/MindTheGapps/vendor_gapps.git vendor/gapps

# Integrasi KernelSU secara otomatis
echo ">>> Mengintegrasikan KernelSU..."
KERNEL_DIR=$(find kernel device -maxdepth 4 -type f -name "Makefile" -exec grep -l "VERSION =" {} + | xargs dirname | head -n 1)
if [ ! -z "$KERNEL_DIR" ]; then
    echo "Ditemukan direktori kernel di: $KERNEL_DIR"
    cd $KERNEL_DIR
    curl -LSs "https://raw.githubusercontent.com/tiann/KernelSU/main/kernel/setup.sh" | bash -
    cd -
else
    echo "Peringatan: Direktori kernel tidak ditemukan. Anda mungkin harus memasukkan KernelSU secara manual."
fi

In [ ]:
%%bash
# 5. Proses Kompilasi / Build ISO dengan GApps disertakan
cd '/content/drive/MyDrive/lineageos_x86'
source build/envsetup.sh

# Flag khusus untuk menyuruh LineageOS menyertakan Google Services
export WITH_GAPPS=true

lunch lineage_x86-userdebug
mka iso_img -j$(nproc --all)

echo "SELESAI! File ISO (yang sudah berisi Google Play Services & KernelSU) akan berada di Google Drive Anda."